# Financial Analytics — Group Project
## Stocks: SBUX, PEP, TSLA, AMZN, HESAY, HOOD
### Data Period: 2023–2025

---
## Section 0: Setup & Configuration

In [ ]:
# Install dependencies (run once)
# !pip install -r requirements.txt

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Apply consistent plot style
from config import (
    TICKERS, START_DATE, END_DATE,
    ROLLING_WINDOW, ACF_LAGS,
    VAR_LEVELS,
    HOLDING_PERIOD, RISK_FREE_RATE, FRONTIER_POINTS,
    PLOT_STYLE,
)

plt.rcParams.update(PLOT_STYLE)

# Import project modules
import data_loader
import eda
import volatility_models
import risk_measures
import portfolio

print(f"Tickers: {TICKERS}")
print(f"Period:  {START_DATE} to {END_DATE}")

---
## Section 1: Data Loading

In [ ]:
# Load data for each stock
stocks = {}
for ticker in TICKERS:
    price, log_ret = data_loader.load_single_stock(ticker)
    stocks[ticker] = (price, log_ret)
    print(f"{ticker}: {len(price)} price obs, {len(log_ret)} return obs")

In [ ]:
# Load multi-stock data for portfolio analysis
prices_df, returns_df = data_loader.load_multiple_stocks(TICKERS)
print(f"Multi-stock data: {prices_df.shape[0]} trading days, {prices_df.shape[1]} assets")
print(f"Date range: {prices_df.index[0].date()} to {prices_df.index[-1].date()}")

---
## Section 2: Exploratory Data Analysis
*Maps to Presentation Section 2: EDA*

In [ ]:
# Price and return time-series for each stock
for ticker in TICKERS:
    price, log_ret = stocks[ticker]
    eda.plot_price_and_returns(price, log_ret, ticker)
    plt.show()

In [ ]:
# Distribution diagnostics (histogram, ACF, QQ)
for ticker in TICKERS:
    _, log_ret = stocks[ticker]
    eda.plot_distribution_diagnostics(log_ret, ticker)
    plt.show()

In [ ]:
# Volatility clustering
for ticker in TICKERS:
    _, log_ret = stocks[ticker]
    eda.plot_volatility_clustering(log_ret, ticker)
    plt.show()

In [ ]:
# Summary statistics comparison table
stats_list = []
for ticker in TICKERS:
    _, log_ret = stocks[ticker]
    s = eda.compute_summary_statistics(log_ret)
    s["ticker"] = ticker
    stats_list.append(s)

summary_df = pd.DataFrame(stats_list).set_index("ticker")
print("=== Summary Statistics (Daily Log-Returns) ===")
display(summary_df)

In [ ]:
# Normality tests
for ticker in TICKERS:
    _, log_ret = stocks[ticker]
    test_df = eda.run_normality_tests(log_ret)
    print(f"\n=== {ticker} — Normality Tests ===")
    display(test_df)

In [ ]:
# ACF of returns and squared returns
for ticker in TICKERS:
    _, log_ret = stocks[ticker]
    eda.plot_acf_returns_and_squared(log_ret, ticker)
    plt.show()

---
## Section 3: Univariate Modeling — Volatility and Risk
*Maps to Presentation Section 3*

### 3a: Volatility Models (GARCH)

In [ ]:
# GARCH model selection and fitting for each stock
analyzers = {}
garch_orders = {}

for ticker in TICKERS:
    _, log_ret = stocks[ticker]
    print(f"\n{'='*60}")
    print(f"Fitting GARCH for {ticker}...")
    print(f"{'='*60}")
    
    best_order, best_result = volatility_models.search_garch(log_ret)
    ar, p, o, q = best_order
    garch_orders[ticker] = best_order
    
    model_type = 'GJR-GARCH' if o > 0 else 'GARCH'
    print(f"Best model: AR({ar}) + {model_type}({p},{o},{q})")
    print(f"BIC: {best_result.bic:.3f}")
    
    analyzer = volatility_models.GARCHAnalyzer(best_result)
    analyzers[ticker] = analyzer
    print(f"nu (t-dist df) = {analyzer.nu:.2f}")

In [ ]:
# GARCH orders comparison
garch_summary = []
for ticker in TICKERS:
    ar, p, o, q = garch_orders[ticker]
    model_type = 'GJR-GARCH' if o > 0 else 'GARCH'
    garch_summary.append({
        'Ticker': ticker,
        'AR': ar, 'p': p, 'o': o, 'q': q,
        'Model': f"AR({ar})+{model_type}({p},{o},{q})",
        'nu': analyzers[ticker].nu,
    })
print("=== GARCH Model Summary ===")
display(pd.DataFrame(garch_summary).set_index('Ticker'))

In [ ]:
# Conditional volatility plots
for ticker in TICKERS:
    _, log_ret = stocks[ticker]
    analyzers[ticker].plot_conditional_volatility(log_ret, ticker)
    plt.show()

In [ ]:
# GARCH residual diagnostics
for ticker in TICKERS:
    analyzers[ticker].plot_residual_diagnostics(ticker)
    plt.show()

### 3b: Static VaR & Expected Shortfall

In [ ]:
# Compute static VaR and ES for each stock
var_tables = {}
es_tables = {}

for ticker in TICKERS:
    _, log_ret = stocks[ticker]
    loss_proxy = -(100 * log_ret)  # percent losses
    
    var_tables[ticker] = risk_measures.compute_var_from_losses(loss_proxy)
    es_tables[ticker] = risk_measures.compute_es_from_losses(loss_proxy)
    
    print(f"\n=== {ticker} — VaR ===")
    display(var_tables[ticker])
    print(f"\n=== {ticker} — ES ===")
    display(es_tables[ticker])

In [ ]:
# VaR comparison plots
for ticker in TICKERS:
    _, log_ret = stocks[ticker]
    loss_proxy = -(100 * log_ret)
    risk_measures.plot_var_comparison(var_tables[ticker], ticker, losses=loss_proxy)
    plt.show()

In [ ]:
# ES comparison plots
for ticker in TICKERS:
    risk_measures.plot_es_comparison(es_tables[ticker], ticker)
    plt.show()

In [ ]:
# Cross-stock VaR comparison at 95% and 99%
for level in [0.95, 0.99]:
    comparison = {}
    for ticker in TICKERS:
        row = var_tables[ticker][var_tables[ticker]['VaR Level'] == level].iloc[0]
        comparison[ticker] = {
            'Historical': row['Historical'],
            'Normal': row['Normal'],
            'T-dist': row['T-dist'],
        }
    print(f"\n=== Cross-Stock VaR at {level:.0%} ===")
    display(pd.DataFrame(comparison).T)

### 3c: Dynamic VaR & ES (GARCH-based)

In [ ]:
# Dynamic VaR and ES for each stock
dyn_var_results = {}
dyn_es_results = {}

for ticker in TICKERS:
    _, log_ret = stocks[ticker]
    loss_proxy = -(100 * log_ret)
    
    mu_r, sigma_r, scale_lam = analyzers[ticker].forecast_series()
    nu = analyzers[ticker].nu
    
    dyn_var = risk_measures.compute_dynamic_var(mu_r, scale_lam, nu)
    dyn_es = risk_measures.compute_dynamic_es(mu_r, scale_lam, nu)
    
    dyn_var_results[ticker] = dyn_var
    dyn_es_results[ticker] = dyn_es
    
    # Plot
    loss_aligned = loss_proxy.reindex(mu_r.index)
    risk_measures.plot_dynamic_risk(loss_aligned, dyn_var, dyn_es, ticker)
    plt.show()
    
    # Exceedance report
    exc_df = risk_measures.count_var_exceedances(loss_aligned, dyn_var)
    print(f"\n=== {ticker} — VaR Exceedances ===")
    display(exc_df)

---
## Section 4: Portfolio Analysis
*Maps to Presentation Section 4*

In [ ]:
# Compute portfolio statistics
port_inputs = portfolio.compute_portfolio_inputs(returns_df)

print(f"=== {HOLDING_PERIOD}-Day Scaled Mean Returns ===")
display(port_inputs['mu_scaled'])

print(f"\n=== {HOLDING_PERIOD}-Day Scaled Standard Deviations ===")
display(port_inputs['std_scaled'])

print(f"\n=== Daily Correlation Matrix ===")
display(port_inputs['corr_daily'])

In [ ]:
# Correlation heatmap
portfolio.plot_correlation_heatmap(port_inputs['corr_daily'], TICKERS)
plt.show()

In [ ]:
# Minimum Variance Portfolio
mvp = portfolio.compute_mvp(
    port_inputs['mu_scaled'],
    port_inputs['cov_scaled'],
    TICKERS,
)

print("=== Minimum Variance Portfolio ===")
print(f"Expected Return ({HOLDING_PERIOD}d): {mvp['expected_return']:.6f}")
print(f"Standard Deviation: {mvp['std_dev']:.6f}")
print(f"\nWeights:")
display(mvp['weights'])

In [ ]:
# Tangency Portfolio
tangency = portfolio.compute_tangency(
    port_inputs['mu_scaled'],
    port_inputs['cov_scaled'],
    rf=RISK_FREE_RATE,
    tickers=TICKERS,
)

print("=== Tangency Portfolio ===")
print(f"Expected Return ({HOLDING_PERIOD}d): {tangency['expected_return']:.6f}")
print(f"Standard Deviation: {tangency['std_dev']:.6f}")
print(f"Sharpe Ratio: {tangency['sharpe_ratio']:.4f}")
print(f"\nWeights:")
display(tangency['weights'])

In [ ]:
# Weight comparison
portfolio.plot_weight_comparison(
    mvp['weights'].values,
    tangency['weights'].values,
    TICKERS,
)
plt.show()

In [ ]:
# Efficient Frontier
frontier_risks, frontier_rets, _ = portfolio.compute_efficient_frontier(
    port_inputs['mu_scaled'],
    port_inputs['cov_scaled'],
)

# Individual asset points
individual_assets = {
    ticker: (float(port_inputs['std_scaled'][ticker]),
             float(port_inputs['mu_scaled'][ticker]))
    for ticker in TICKERS
}

portfolio.plot_efficient_frontier(
    frontier_risks, frontier_rets,
    mvp, tangency,
    individual_assets,
    rf=RISK_FREE_RATE,
    title=f"Efficient Frontier ({HOLDING_PERIOD}-Day Holding Period)",
)
plt.show()

In [ ]:
# Portfolio vs Individual Risk comparison
risk_comparison = portfolio.compare_portfolio_vs_individual_risk(
    port_inputs['std_scaled'].values,
    mvp['std_dev'],
    tangency['std_dev'],
    TICKERS,
)

print("=== Risk Comparison: Individual Assets vs Portfolios ===")
display(risk_comparison)

---
## Section 5: Summary & Cross-Stock Comparison
*Maps to Presentation Section 5: Discussion and Interpretation*

In [ ]:
# Cross-stock summary: key risk metrics
risk_summary = []
for ticker in TICKERS:
    var_99_hist = float(var_tables[ticker][var_tables[ticker]['VaR Level'] == 0.99]['Historical'].iloc[0])
    es_99_hist = float(es_tables[ticker][es_tables[ticker]['ES Level'] == 0.99]['Historical'].iloc[0])
    ar, p, o, q = garch_orders[ticker]
    model_type = 'GJR-GARCH' if o > 0 else 'GARCH'
    
    risk_summary.append({
        'Ticker': ticker,
        'Daily Std (%)': summary_df.loc[ticker, 'std'] * 100,
        'Skewness': summary_df.loc[ticker, 'skewness'],
        'Excess Kurt.': summary_df.loc[ticker, 'excess_kurtosis'],
        'GARCH Model': f"AR({ar})+{model_type}({p},{o},{q})",
        'GARCH nu': analyzers[ticker].nu,
        'VaR 99% (Hist)': var_99_hist,
        'ES 99% (Hist)': es_99_hist,
    })

risk_summary_df = pd.DataFrame(risk_summary).set_index('Ticker')
print("=== Cross-Stock Risk Summary ===")
display(risk_summary_df)

In [ ]:
# Portfolio diversification benefit
print("=== Diversification Benefit ===")
print(f"\nMinimum individual std ({HOLDING_PERIOD}d): "
      f"{port_inputs['std_scaled'].min():.6f} ({port_inputs['std_scaled'].idxmin()})")
print(f"MVP std ({HOLDING_PERIOD}d):                  {mvp['std_dev']:.6f}")
print(f"Tangency std ({HOLDING_PERIOD}d):             {tangency['std_dev']:.6f}")
print(f"\nMVP reduces risk by: "
      f"{(1 - mvp['std_dev'] / port_inputs['std_scaled'].min()) * 100:.1f}% "
      f"vs. least risky individual asset")